# Halden Energy - Week 4 Faculty Solution

This notebook loads the same canonical CSV files as the student notebook and completes the current-week Week 4 analysis. It does not introduce economics beyond the constants ledger, Week 4 spec, and supplied package files.

In [ ]:
from pathlib import Path
from decimal import Decimal
import pandas as pd

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "data").exists() and (PACKAGE_ROOT.parent / "data").exists():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
DATA_DIR = PACKAGE_ROOT / "data"

lift = pd.read_csv(DATA_DIR / "permian_lifting.csv", dtype=str)
const = pd.read_csv(DATA_DIR / "cost_constants.csv", dtype=str)
comp = pd.read_csv(DATA_DIR / "segment_comp.csv", dtype=str)

def cv(name):
    return Decimal(const.loc[const["parameter"] == name, "value"].iloc[0])

def money(value):
    return value.quantize(Decimal("0.00"))

def out(value):
    return str(money(value))

In [ ]:
wti = cv("wti")
disc = cv("permian_wellhead_discount")
transport = cv("transport_permian_to_br")
gathering = cv("gathering_cost")
srcap = cv("sr_capital_charge")
crack = cv("gc_crack_321")
complexity = cv("br_complexity_premium")
opex = cv("br_opex")
capture_rate = cv("geneva_capture_rate")
max_volume = cv("geneva_max_volume")

lift_2024 = Decimal(lift.loc[lift["vintage"] == "2024", "cash_lifting_cost_usd_bbl"].iloc[0])
up_target = Decimal(comp.loc[comp["segment"] == "upstream", "target_margin_usd_bbl"].iloc[0])
ref_target = Decimal(comp.loc[comp["segment"] == "refining", "target_margin_usd_bbl"].iloc[0])

delivered_marginal_cost = lift_2024 + gathering + transport
realized_wellhead = wti - disc
product_slate_value = realized_wellhead + crack + complexity
integrated_margin = product_slate_value - delivered_marginal_cost - opex

tp_market = realized_wellhead + transport
tp_marginal = delivered_marginal_cost + srcap
tp_midpoint = (tp_market + tp_marginal) / Decimal("2")

def split(tp):
    upstream = tp - delivered_marginal_cost
    refining = product_slate_value - tp - opex
    return {
        "transfer_price": out(tp),
        "upstream_margin": out(upstream),
        "refining_margin": out(refining),
        "upstream_vs_target": out(upstream - up_target),
        "refining_vs_target": out(refining - ref_target),
        "integrated_margin": out(upstream + refining),
    }

geneva_gap = tp_market - tp_midpoint
geneva_capture_per_bbl = geneva_gap * capture_rate

reference_outputs = {
    "delivered_marginal_cost": out(delivered_marginal_cost),
    "transfer_prices": {
        "market": out(tp_market),
        "marginal_cost": out(tp_marginal),
        "lazy_midpoint": out(tp_midpoint),
    },
    "integrated_margin": out(integrated_margin),
    "segment_splits": {
        "market": split(tp_market),
        "marginal_cost": split(tp_marginal),
        "lazy_midpoint": split(tp_midpoint),
    },
    "geneva_arbitrage": {
        "gap": out(geneva_gap),
        "capture_rate": str(capture_rate),
        "capture_per_bbl": str(geneva_capture_per_bbl.normalize()),
        "max_volume_bbl_day": int(max_volume),
        "time_basis": "unresolved; do not annualize or monthly-convert without an authoritative rule",
    },
}

reference_outputs

In [ ]:
assert reference_outputs["delivered_marginal_cost"] == "14.10"
assert reference_outputs["transfer_prices"] == {
    "market": "73.70",
    "marginal_cost": "18.70",
    "lazy_midpoint": "46.20",
}
assert reference_outputs["integrated_margin"] == "76.75"
assert reference_outputs["segment_splits"]["market"]["upstream_margin"] == "59.60"
assert reference_outputs["segment_splits"]["market"]["refining_margin"] == "17.15"
assert reference_outputs["segment_splits"]["marginal_cost"]["upstream_margin"] == "4.60"
assert reference_outputs["segment_splits"]["marginal_cost"]["refining_margin"] == "72.15"
assert reference_outputs["segment_splits"]["lazy_midpoint"]["upstream_margin"] == "32.10"
assert reference_outputs["segment_splits"]["lazy_midpoint"]["refining_margin"] == "44.65"
assert reference_outputs["geneva_arbitrage"]["gap"] == "27.50"
assert reference_outputs["geneva_arbitrage"]["capture_per_bbl"] == "9.625"
print("Week 4 faculty solution checks passed")